<a href="https://colab.research.google.com/github/OkyereBiew/bayesian-credit-risk-simulator/blob/main/notebooks/04_risk_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
column_names = [
    "checking_account",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_duration",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "target"
]

df = pd.read_csv(
    "german.data",
    sep=" ",
    header=None,
    names=column_names
)

df["target"] = df["target"].map({
    1: 0,
    2: 1
})

categorical_columns = df.select_dtypes(include=["object"]).columns

label_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

X = df.drop("target", axis=1)

y = df["target"]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
with pm.Model() as bayesian_logistic_model:

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_scaled.shape[1]
    )

    alpha = pm.Normal(
        "alpha",
        mu=0,
        sigma=1
    )

    mu = alpha + pm.math.dot(X_scaled, beta)

    theta = pm.Deterministic(
        "theta",
        pm.math.sigmoid(mu)
    )

    likelihood = pm.Bernoulli(
        "likelihood",
        p=theta,
        observed=y
    )

    trace = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        random_seed=42
    )

In [ ]:
posterior_probs = trace.posterior["theta"].mean(
    dim=["chain", "draw"]
).values

posterior_probs[:10]

In [ ]:
risk_results = pd.DataFrame({
    "Default_Probability": posterior_probs,
    "Actual_Target": y
})

risk_results.head()

In [ ]:
def categorize_risk(probability):

    if probability < 0.30:
        return "Low Risk"

    elif probability < 0.60:
        return "Medium Risk"

    else:
        return "High Risk"

risk_results["Risk_Category"] = risk_results[
    "Default_Probability"
].apply(categorize_risk)

risk_results.head()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=risk_results,
    x="Risk_Category"
)

plt.title("Borrower Risk Category Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Count")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    risk_results["Default_Probability"],
    bins=30,
    kde=True
)

plt.title("Posterior Default Probability Distribution")
plt.xlabel("Default Probability")
plt.ylabel("Frequency")

plt.show()

In [ ]:
high_risk = risk_results[
    risk_results["Risk_Category"] == "High Risk"
]

high_risk.head()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=risk_results,
    x="Risk_Category",
    y="Default_Probability"
)

plt.title("Default Probability by Risk Category")
plt.xlabel("Risk Category")
plt.ylabel("Default Probability")

plt.show()

## Probabilistic Risk Analysis

The Bayesian credit risk model generates posterior default probabilities for individual borrowers rather than producing only binary classifications.

This allows uncertainty-aware risk assessment and supports more interpretable financial decision-making.

Borrowers were categorized into low-, medium-, and high-risk groups based on posterior default probabilities. Such probabilistic segmentation is commonly used in financial risk management and lending analytics.